# Heat Pump Grid Load Forecasting — v2
**Improved extrapolation:** summer-specific model + seasonal ratio features + autoregressive LSTM

In [1]:
!pip install lightgbm xgboost scikit-learn pandas numpy matplotlib seaborn tqdm

In [2]:
# 1. Install gdown 
!pip install -q gdown

# 2. Define the ID and download
import gdown

file_id = '13kg33hSj4r-NIHwaXLuTzNQjaOVWcmIh'
url = f'https://drive.google.com/uc?id={file_id}'

# This downloads the file to your current working directory
gdown.download(url, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=13kg33hSj4r-NIHwaXLuTzNQjaOVWcmIh
From (redirected): https://drive.google.com/uc?id=13kg33hSj4r-NIHwaXLuTzNQjaOVWcmIh&confirm=t&uuid=1bcd23fa-5108-4b64-9a1b-3c315cda99fa
To: /kaggle/working/ensemble_task_3_data_oct24-oct25.tar.gz
100%|██████████| 437M/437M [00:04<00:00, 97.8MB/s] 


'ensemble_task_3_data_oct24-oct25.tar.gz'

In [3]:
import tarfile

# Update with your actual filename
filename = "/kaggle/working/ensemble_task_3_data_oct24-oct25.tar.gz"

with tarfile.open(filename, "r:gz") as tar:
    tar.extractall()
    print("Files extracted successfully!")
    
# Verify the files are there
import os
print(os.listdir('.'))

/tmp/ipykernel_55/3777795485.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Files extracted successfully!
['data.csv', '.virtual_documents', 'devices.csv', 'ensemble_task_3_data_oct24-oct25.tar.gz']


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, gc, importlib, threading
from pathlib import Path
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
np.random.seed(42)

DATA_DIR    = Path('.')
DATA_CSV    = DATA_DIR / 'data.csv'
DEVICES_CSV = DATA_DIR / 'devices.csv'
OUTPUT_CSV  = DATA_DIR / 'submission.csv'

print('Libraries loaded.')
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())

Libraries loaded.
CUDA available: True
GPU count: 2


## 1. Data Ingestion

In [5]:
devices = pd.read_csv(DEVICES_CSV)
print(f'Devices: {devices.shape}')
devices.head()

Devices: (600, 3)


,latitude,longitude,deviceId
0,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...
1,53.5,21.1,005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30f...
2,52.9,18.1,01668c64ccc16c506a7c1a5c032e2eb5e2de48ecb284f2...
3,52.5,17.7,01bf745bf2df0312bd5ff2234c0e9dedc39ad0bac9bcfc...
4,50.7,16.7,02e4ad5d8d0016d35a003ea6df7e10fe27093aba81c64a...


In [6]:
CHUNKSIZE = 2_000_000
TEMP_COLS = [f't{i}' for i in range(1, 14)]
KEEP_COLS = ['deviceId', 'timedate', 'period', 'deviceType', 'x1', 'x2', 'x3'] + TEMP_COLS
DTYPE_MAP = {col: 'float32' for col in TEMP_COLS + ['x1', 'x2']}
DTYPE_MAP.update({'x3': 'int16', 'deviceType': 'int16'})

agg_funcs = {col: 'mean' for col in TEMP_COLS}
agg_funcs['x1'] = 'mean'
agg_funcs['x2'] = ['mean', 'count']
agg_funcs['x3'] = 'first'

def parse_year_month(df):
    td = df['timedate'].str
    df['year']  = td[0:4].astype('int16')
    df['month'] = td[5:7].astype('int8')
    return df

def aggregate_chunk(chunk):
    daily = chunk.groupby(['deviceId','deviceType','period','year','month'], sort=False).agg(agg_funcs)
    daily.columns = ['_'.join(c) if isinstance(c, tuple) else c for c in daily.columns]
    return daily.reset_index()

agg_chunks = []
print('Reading data in chunks...')
reader = pd.read_csv(DATA_CSV, usecols=KEEP_COLS, dtype=DTYPE_MAP, chunksize=CHUNKSIZE, low_memory=False)
for chunk in tqdm(reader):
    chunk = parse_year_month(chunk)
    chunk.drop(columns=['timedate'], inplace=True)
    agg_chunks.append(aggregate_chunk(chunk))
    del chunk; gc.collect()

print('Concatenating...')
daily_df = pd.concat(agg_chunks, ignore_index=True)
del agg_chunks; gc.collect()

re_agg = {c: 'mean' for c in daily_df.columns
          if c not in ['deviceId','deviceType','period','year','month','x2_count','x3_first']}
re_agg.update({'x2_count':'sum','x3_first':'first','deviceType':'first','period':'first'})
daily_df = daily_df.groupby(['deviceId','year','month'], sort=False).agg(re_agg).reset_index()
gc.collect()

print(f'Done. Shape: {daily_df.shape}')
print('Period counts:', daily_df['period'].value_counts().to_dict())
daily_df.head()

Reading data in chunks...


33it [04:08,  7.52s/it]


Concatenating...
Done. Shape: (7701, 22)
Period counts: {'train': 4142, 'test': 2363, 'valid': 1196}


,deviceId,year,month,t1_mean,t2_mean,t3_mean,t4_mean,t5_mean,t6_mean,t7_mean,t8_mean,t9_mean,t10_mean,t11_mean,t12_mean,t13_mean,x1_mean,x2_mean,x2_count,x3_first,deviceType,period
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024,10,0.311391,0.049598,0.0,0.390205,0.440752,0.430313,0.206753,0.511103,0.380923,0.210840,0.209803,0.066749,0.068050,0.0,0.096346,8914,8,19,train
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024,11,0.274585,0.043373,0.0,0.426048,0.472507,0.457352,0.206652,0.518456,0.413211,0.206846,0.204807,0.070019,0.072996,0.0,0.206957,8615,8,19,train
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024,12,0.266031,0.040000,0.0,0.441585,0.481060,0.464548,0.206692,0.524291,0.420874,0.204308,0.202504,0.070345,0.074770,0.0,0.241164,8921,8,19,train
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,1,0.263986,0.040000,0.0,0.452613,0.485823,0.467861,0.206773,0.528462,0.425033,0.203875,0.202686,0.070920,0.075966,0.0,0.265698,8733,8,19,train
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,2,0.258883,0.040000,0.0,0.456488,0.487035,0.468585,0.206657,0.532827,0.426108,0.203268,0.202415,0.070691,0.076328,0.0,0.283868,8055,8,19,train


In [7]:
daily_df = daily_df.merge(devices[['deviceId','latitude','longitude']], on='deviceId', how='left')
print('Unique devices:', daily_df['deviceId'].nunique())
print('Period counts:', daily_df['period'].value_counts().to_dict())

Unique devices: 600
Period counts: {'train': 4142, 'test': 2363, 'valid': 1196}


## 2. Monthly Aggregation

In [8]:
monthly_agg = {}
for col in TEMP_COLS:
    mean_col = f'{col}_mean'
    if mean_col in daily_df.columns:
        monthly_agg[mean_col] = 'mean'
for col in ['x1_mean']:
    if col in daily_df.columns:
        monthly_agg[col] = 'mean'
monthly_agg.update({
    'x2_mean': 'mean', 'x2_count': 'sum', 'x3_first': 'first',
    'deviceType': 'first', 'latitude': 'first', 'longitude': 'first', 'period': 'first',
})

monthly_df = daily_df.groupby(['deviceId','year','month']).agg(monthly_agg).reset_index()
monthly_df.rename(columns={'x2_mean': 'target'}, inplace=True)
# x2 withheld for valid/test — null target to avoid leakage
monthly_df.loc[monthly_df['period'] != 'train', 'target'] = np.nan

print('Monthly shape:', monthly_df.shape)
print('Period counts:', monthly_df['period'].value_counts().to_dict())
monthly_df.head()

Monthly shape: (7701, 24)
Period counts: {'train': 4142, 'test': 2363, 'valid': 1196}


,deviceId,year,month,t1_mean,t2_mean,t3_mean,t4_mean,t5_mean,t6_mean,t7_mean,t8_mean,t9_mean,t10_mean,t11_mean,t12_mean,t13_mean,x1_mean,target,x2_count,x3_first,deviceType,latitude,longitude,period
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024,10,0.311391,0.049598,0.0,0.390205,0.440752,0.430313,0.206753,0.511103,0.380923,0.210840,0.209803,0.066749,0.068050,0.0,0.096346,8914,8,19,50.0,18.3,train
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024,11,0.274585,0.043373,0.0,0.426048,0.472507,0.457352,0.206652,0.518456,0.413211,0.206846,0.204807,0.070019,0.072996,0.0,0.206957,8615,8,19,50.0,18.3,train
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024,12,0.266031,0.040000,0.0,0.441585,0.481060,0.464548,0.206692,0.524291,0.420874,0.204308,0.202504,0.070345,0.074770,0.0,0.241164,8921,8,19,50.0,18.3,train
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,1,0.263986,0.040000,0.0,0.452613,0.485823,0.467861,0.206773,0.528462,0.425033,0.203875,0.202686,0.070920,0.075966,0.0,0.265698,8733,8,19,50.0,18.3,train
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,2,0.258883,0.040000,0.0,0.456488,0.487035,0.468585,0.206657,0.532827,0.426108,0.203268,0.202415,0.070691,0.076328,0.0,0.283868,8055,8,19,50.0,18.3,train


## 3. Feature Engineering

In [9]:
# ── Geo clustering ────────────────────────────────────────────────────────────
geo = monthly_df[['deviceId','latitude','longitude']].drop_duplicates('deviceId').dropna()
km = KMeans(n_clusters=6, random_state=42, n_init=10)
geo['geo_cluster'] = km.fit_predict(geo[['latitude','longitude']])
monthly_df = monthly_df.merge(geo[['deviceId','geo_cluster']], on='deviceId', how='left')

# ── Encode categoricals ───────────────────────────────────────────────────────
for col in ['deviceType','x3_first','geo_cluster']:
    if col in monthly_df.columns:
        monthly_df[col] = monthly_df[col].fillna(-1).astype(int)
monthly_df['device_enc'] = LabelEncoder().fit_transform(monthly_df['deviceId'])

# ── Temporal features ─────────────────────────────────────────────────────────
monthly_df['month_sin'] = np.sin(2 * np.pi * monthly_df['month'] / 12)
monthly_df['month_cos'] = np.cos(2 * np.pi * monthly_df['month'] / 12)
monthly_df['is_summer'] = monthly_df['month'].isin([5,6,7,8,9,10]).astype(int)
monthly_df['is_winter'] = monthly_df['month'].isin([11,12,1,2,3]).astype(int)
print('Temporal features added.')

Temporal features added.


In [10]:
# ── Lag & seasonal features ───────────────────────────────────────────────────
monthly_df = monthly_df.sort_values(['deviceId','year','month']).reset_index(drop=True)
monthly_df['ym'] = monthly_df['year'] * 12 + monthly_df['month']

for lag in [1, 2, 3, 6]:
    monthly_df[f'target_lag{lag}'] = monthly_df.groupby('deviceId')['target'].shift(lag)

monthly_df['target_roll3'] = (
    monthly_df.groupby('deviceId')['target']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

device_stats = (
    monthly_df[monthly_df['target'].notna()]
    .groupby('deviceId')['target']
    .agg(device_mean='mean', device_std='std').reset_index()
)
monthly_df = monthly_df.merge(device_stats, on='deviceId', how='left')

seasonal = (
    monthly_df[monthly_df['target'].notna()]
    .groupby(['deviceId','month'])['target'].mean()
    .rename('seasonal_idx').reset_index()
)
monthly_df = monthly_df.merge(seasonal, on=['deviceId','month'], how='left')

cluster_seasonal = (
    monthly_df[monthly_df['target'].notna()]
    .groupby(['geo_cluster','month'])['target'].mean()
    .rename('cluster_seasonal_idx').reset_index()
)
monthly_df = monthly_df.merge(cluster_seasonal, on=['geo_cluster','month'], how='left')
monthly_df['seasonal_idx'] = monthly_df['seasonal_idx'].fillna(monthly_df['cluster_seasonal_idx'])

# ── NEW: t1 trend (is external temp rising or falling?) ───────────────────────
monthly_df['t1_lag1']  = monthly_df.groupby('deviceId')['t1_mean'].shift(1)
monthly_df['t1_trend'] = monthly_df['t1_mean'] - monthly_df['t1_lag1']

# ── NEW: seasonal ratio (Oct x2 / winter mean — proxy for summer behaviour) ───
oct_mean    = monthly_df[(monthly_df['month']==10) & (monthly_df['period']=='train')].groupby('deviceId')['target'].mean()
winter_mean = monthly_df[monthly_df['period']=='train'].groupby('deviceId')['target'].mean()
summer_ratio = (oct_mean / winter_mean.replace(0, np.nan)).fillna(1.0)
monthly_df['summer_ratio'] = monthly_df['deviceId'].map(summer_ratio).fillna(1.0)

print('All features added.')
print('Columns:', [c for c in monthly_df.columns if c not in ['deviceId','latitude','longitude']])

All features added.
Columns: ['year', 'month', 't1_mean', 't2_mean', 't3_mean', 't4_mean', 't5_mean', 't6_mean', 't7_mean', 't8_mean', 't9_mean', 't10_mean', 't11_mean', 't12_mean', 't13_mean', 'x1_mean', 'target', 'x2_count', 'x3_first', 'deviceType', 'period', 'geo_cluster', 'device_enc', 'month_sin', 'month_cos', 'is_summer', 'is_winter', 'ym', 'target_lag1', 'target_lag2', 'target_lag3', 'target_lag6', 'target_roll3', 'device_mean', 'device_std', 'seasonal_idx', 'cluster_seasonal_idx', 't1_lag1', 't1_trend', 'summer_ratio']


## 4. Train / Val / Test Split

In [11]:
train_mask = monthly_df['period'] == 'train'
val_mask   = monthly_df['period'] == 'valid'
test_mask  = monthly_df['period'] == 'test'

if train_mask.sum() == 0:
    print('WARNING: falling back to date-based splits')
    train_mask = (((monthly_df['year']==2024)&(monthly_df['month']>=10))|
                  ((monthly_df['year']==2025)&(monthly_df['month']<=4)))
    val_mask   = (monthly_df['year']==2025)&(monthly_df['month'].isin([5,6]))
    test_mask  = (monthly_df['year']==2025)&(monthly_df['month'].isin([7,8,9,10]))

print(f'Train: {train_mask.sum()}  Val: {val_mask.sum()}  Test: {test_mask.sum()}')

EXCLUDE  = {'deviceId','target','ym','latitude','longitude','cluster_seasonal_idx','period'}
FEATURES = [c for c in monthly_df.columns if c not in EXCLUDE]

# Drop zero-variance features
zero_var = [c for c in FEATURES if monthly_df.loc[train_mask, c].std() == 0]
print(f'Dropping zero-variance features: {zero_var}')
FEATURES = [c for c in FEATURES if c not in zero_var]
print(f'Features ({len(FEATURES)}): {FEATURES}')

X_train = monthly_df.loc[train_mask, FEATURES].fillna(0).reset_index(drop=True)
y_train = monthly_df.loc[train_mask, 'target'].reset_index(drop=True)
X_val   = monthly_df.loc[val_mask,   FEATURES].fillna(0).reset_index(drop=True)
X_test  = monthly_df.loc[test_mask,  FEATURES].fillna(0).reset_index(drop=True)

X_tr, X_ev, y_tr, y_ev = train_test_split(X_train, y_train, test_size=0.15, random_state=42)
X_tr = X_tr.reset_index(drop=True); X_ev = X_ev.reset_index(drop=True)
y_tr = y_tr.reset_index(drop=True); y_ev = y_ev.reset_index(drop=True)

print(f'X_tr: {X_tr.shape}  X_ev: {X_ev.shape}')
print(f'X_val: {X_val.shape}  X_test: {X_test.shape}')

Train: 4142  Val: 1196  Test: 2363
Dropping zero-variance features: []
Features (36): ['year', 'month', 't1_mean', 't2_mean', 't3_mean', 't4_mean', 't5_mean', 't6_mean', 't7_mean', 't8_mean', 't9_mean', 't10_mean', 't11_mean', 't12_mean', 't13_mean', 'x1_mean', 'x2_count', 'x3_first', 'deviceType', 'geo_cluster', 'device_enc', 'month_sin', 'month_cos', 'is_summer', 'is_winter', 'target_lag1', 'target_lag2', 'target_lag3', 'target_lag6', 'target_roll3', 'device_mean', 'device_std', 'seasonal_idx', 't1_lag1', 't1_trend', 'summer_ratio']
X_tr: (3520, 36)  X_ev: (622, 36)
X_val: (1196, 36)  X_test: (2363, 36)


## 5. Tree Models (GPU parallel)

In [12]:
import subprocess
result = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                       capture_output=True, text=True)
for i, line in enumerate(result.stdout.strip().split('\n')):
    print(f'GPU {i}: {line}')

GPU 0: Tesla T4, 15360 MiB
GPU 1: Tesla T4, 15360 MiB


In [13]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'device': 'gpu',
        'gpu_device_id': 0,
        'verbose': -1,
        'random_state': 42,
        # Parameters to tune
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'lambda_l1':         trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2':         trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
        'n_estimators': 3000,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_ev, y_ev)],
          callbacks=[lgb.early_stopping(100, verbose=False),
                     lgb.log_evaluation(-1)])
    return mean_absolute_error(y_ev, m.predict(X_ev))

study_lgb = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'Best LGB MAE: {study_lgb.best_value:.6f}')
print(f'Best LGB params: {study_lgb.best_params}')

  0%|          | 0/50 [00:00<?, ?it/s]

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Best LGB MAE: 0.001110
Best LGB params: {'num_leaves': 131, 'learning_rate': 0.01906444632168304, 'feature_fraction': 0.9644572330400879, 'bagging_fraction': 0.870421199140574, 'bagging_freq': 10, 'min_child_samples': 7, 'lambda_l1': 0.0003166823369361092, 'lambda_l2': 0.001577934369498659}


In [14]:
def objective_xgb(trial):
    params = {
        'objective': 'reg:absoluteerror',
        'eval_metric': 'mae',
        'tree_method': 'hist',
        'device': 'cuda:1',
        'verbosity': 0,
        'random_state': 43,
        'early_stopping_rounds': 100,
        # Parameters to tune
        'max_depth':        trial.suggest_int('max_depth', 4, 12),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
        'alpha':            trial.suggest_float('alpha', 1e-4, 10.0, log=True),
        'lambda':           trial.suggest_float('lambda', 1e-4, 10.0, log=True),
        'n_estimators': 3000,
    }
    m = xgb.XGBRegressor(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_ev, y_ev)], verbose=False)
    return mean_absolute_error(y_ev, m.predict(X_ev))

study_xgb = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=43))
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print(f'Best XGB MAE: {study_xgb.best_value:.6f}')
print(f'Best XGB params: {study_xgb.best_params}')

  0%|          | 0/50 [00:00<?, ?it/s]

Best XGB MAE: 0.001130
Best XGB params: {'max_depth': 4, 'learning_rate': 0.008713035938332116, 'subsample': 0.6122405338441916, 'colsample_bytree': 0.9803961436025844, 'min_child_weight': 10, 'alpha': 0.0003487155359758279, 'lambda': 5.469122633970875}


In [15]:
# Retrain with best found params on full training data
best_lgb_params = {**study_lgb.best_params,
                   'objective': 'regression_l1', 'metric': 'mae',
                   'device': 'gpu', 'gpu_device_id': 0,
                   'verbose': -1, 'random_state': 42, 'n_estimators': 6000}

best_xgb_params = {**study_xgb.best_params,
                   'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
                   'tree_method': 'hist', 'device': 'cuda:1',
                   'verbosity': 0, 'random_state': 43,
                   'early_stopping_rounds': 150, 'n_estimators': 6000}

# Retrain in parallel with best params
lgb_params = best_lgb_params
xgb_params = best_xgb_params
# ... then run your parallel training cell as before

In [16]:
#lgb_params = {
 #   'objective': 'regression_l1', 'metric': 'mae',
  #  'device': 'gpu', 'gpu_device_id': 0,
   # 'num_leaves': 127, 'learning_rate': 0.01,
    #'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    #'min_child_samples': 20, 'n_estimators': 10000, 'verbose': -1, 'random_state': 42,
#}
#xgb_params = {
 #   'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
  #  'tree_method': 'hist', 'device': 'cuda:1',
   # 'max_depth': 8, 'learning_rate': 0.01,
    #'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 10,
    #'n_estimators': 10000, 'early_stopping_rounds': 1000, 'verbosity': 0, 'random_state': 43,
#}

lgb_results, xgb_results = {}, {}

def train_lgb():
    try:
        m = lgb.LGBMRegressor(**lgb_params)
        m.fit(X_tr, y_tr, eval_set=[(X_ev, y_ev)],
              callbacks=[lgb.early_stopping(1000, verbose=False), lgb.log_evaluation(500)])
        lgb_results['model'] = m
        lgb_results['val']   = m.predict(X_val)
        lgb_results['test']  = m.predict(X_test)
        print(f'[LGB done] best iter: {m.best_iteration_}')
    except Exception as e:
        import traceback; lgb_results['error'] = traceback.format_exc(); print(f'[LGB ERROR] {e}')

def train_xgb():
    try:
        m = xgb.XGBRegressor(**xgb_params)
        m.fit(X_tr, y_tr, eval_set=[(X_ev, y_ev)], verbose=500)
        xgb_results['model'] = m
        xgb_results['val']   = m.predict(X_val)
        xgb_results['test']  = m.predict(X_test)
        print(f'[XGB done] best iter: {m.best_iteration}')
    except Exception as e:
        import traceback; xgb_results['error'] = traceback.format_exc(); print(f'[XGB ERROR] {e}')

print('Training LightGBM (GPU 0) + XGBoost (GPU 1) in parallel...')
t1 = threading.Thread(target=train_lgb)
t2 = threading.Thread(target=train_xgb)
t1.start(); t2.start()
t1.join();  t2.join()
print('Both done.')

if 'error' in lgb_results: print(lgb_results['error']); raise RuntimeError('LightGBM failed')
if 'error' in xgb_results: print(xgb_results['error']); raise RuntimeError('XGBoost failed')

lgb_model = lgb_results['model']; lgb_val_pred = lgb_results['val']; lgb_test_pred = lgb_results['test']
xgb_model = xgb_results['model']; xgb_val_pred = xgb_results['val']; xgb_test_pred = xgb_results['test']

lgb_mae = mean_absolute_error(y_ev, lgb_model.predict(X_ev))
xgb_mae = mean_absolute_error(y_ev, xgb_model.predict(X_ev))
print(f'LightGBM internal eval MAE: {lgb_mae:.6f}')
print(f'XGBoost  internal eval MAE: {xgb_mae:.6f}')

Training LightGBM (GPU 0) + XGBoost (GPU 1) in parallel...
[0]	validation_0-mae:0.09246
[500]	validation_0-mae:0.00261
[1000]	validation_0-mae:0.00127
[1500]	validation_0-mae:0.00117
[2000]	validation_0-mae:0.00115
[2500]	validation_0-mae:0.00114
[500]	valid_0's l1: 0.00111309
[3000]	validation_0-mae:0.00113
[3500]	validation_0-mae:0.00113
[4000]	validation_0-mae:0.00112
[4500]	validation_0-mae:0.00112
[4834]	validation_0-mae:0.00112
[XGB done] best iter: 4684
[1000]	valid_0's l1: 0.00111439
[1500]	valid_0's l1: 0.00111679
[LGB done] best iter: 529
Both done.
LightGBM internal eval MAE: 0.001110
XGBoost  internal eval MAE: 0.001118


In [17]:
train_cols = set(X_tr.columns)
val_cols = set(X_val.columns)

print("In Val but NOT in Train:", val_cols - train_cols)
print("In Train but NOT in Val:", train_cols - val_cols)

In Val but NOT in Train: set()
In Train but NOT in Val: set()


In [18]:
W_LGB    = 0.5
W_XGB    = 0.5



blend_val  = (W_LGB  * lgb_val_pred
            + W_XGB  * xgb_val_pred).clip(0, 1)

blend_test = (W_LGB  * lgb_test_pred
            + W_XGB  * xgb_test_pred).clip(0, 1)

blend_mae = mean_absolute_error(y_ev,
    W_LGB  * lgb_model.predict(X_ev) +
    W_XGB  * xgb_model.predict(X_ev))
print(f'Blended internal eval MAE: {blend_mae:.6f}')

Blended internal eval MAE: 0.000832


## 10. Submission

In [19]:
val_sub  = monthly_df.loc[val_mask,  ['deviceId','year','month']].copy()
test_sub = monthly_df.loc[test_mask, ['deviceId','year','month']].copy()
val_sub['prediction']  = blend_val
test_sub['prediction'] = blend_test

submission = pd.concat([val_sub, test_sub], ignore_index=True)
submission = submission.sort_values(['deviceId','year','month']).reset_index(drop=True)
submission['prediction'] = submission['prediction'].clip(0, 1)

print('Submission shape:', submission.shape)
print(submission['prediction'].describe())
submission.head(12)

Submission shape: (3559, 4)
count    3559.000000
mean        0.016551
std         0.038080
min         0.000647
25%         0.004349
50%         0.004775
75%         0.005595
max         0.465972
Name: prediction, dtype: float64


,deviceId,year,month,prediction
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,5,0.005346
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,6,0.005168
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,7,0.005194
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,8,0.005218
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,9,0.005252
5,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2025,10,0.096624
6,005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30f...,2025,5,0.004961
7,005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30f...,2025,6,0.004674
8,005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30f...,2025,7,0.004663
9,005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30f...,2025,8,0.004744


In [20]:
# Expand to ALL devices in raw data (some only appear in test period)
print('Scanning raw data.csv for all deviceIds...')
device_chunks = []
for chunk in pd.read_csv(DATA_CSV, usecols=['deviceId'], chunksize=2_000_000):
    device_chunks.append(chunk['deviceId'].unique())
all_devices_raw = np.unique(np.concatenate(device_chunks))
print(f'Total unique devices: {len(all_devices_raw)}')
print(f'Expected submission rows: {len(all_devices_raw) * 6}')

full_grid = pd.DataFrame([
    {'deviceId': dev, 'year': 2025, 'month': m}
    for dev in all_devices_raw for m in [5,6,7,8,9,10]
])
full_grid = full_grid.merge(submission[['deviceId','year','month','prediction']],
                            on=['deviceId','year','month'], how='left')

device_means = monthly_df[monthly_df['period']=='train'].groupby('deviceId')['target'].mean()
global_mean  = monthly_df[monthly_df['period']=='train']['target'].mean()
print(f'Global mean fallback: {global_mean:.6f}')

full_grid['prediction'] = full_grid.apply(
    lambda r: r['prediction'] if pd.notna(r['prediction'])
              else device_means.get(r['deviceId'], global_mean), axis=1
).clip(0, 1)

assert not full_grid['prediction'].isna().any(), 'NaN predictions!'
print(f'Final rows: {len(full_grid)}  Missing: {full_grid["prediction"].isna().sum()}')

full_grid.to_csv('submission_full.csv', index=False)
submission.to_csv(OUTPUT_CSV, index=False)
print('Saved submission_full.csv and submission.csv')
print(full_grid.head(12).to_string(index=False))

Scanning raw data.csv for all deviceIds...
Total unique devices: 600
Expected submission rows: 3600
Global mean fallback: 0.158669
Final rows: 3600  Missing: 0
Saved submission_full.csv and submission.csv
                                                        deviceId  year  month  prediction
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025      5    0.005346
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025      6    0.005168
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025      7    0.005194
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025      8    0.005218
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025      9    0.005252
000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc93ae46eecc5798f0fe  2025     10    0.096624
005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30fe5a99d7a76b53f9fc9  2025      5    0.004961
005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30fe5a99d7a76b53f9fc9  2025     